In [1]:
import sys

sys.path.append("..")

In [2]:
import os
from src.api.dependencies.injectables import (
    get_mongo_vdb,
    get_chat_model,
    get_topic_selector,
    get_rag_engine,
    get_topic_prompt_builder,
    get_embedding_model,
    get_agent,
)
from src.mongo import get_mongo_db
from src import ENV
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from IPython.display import display, Markdown
from langchain_core.messages import AIMessage, HumanMessage


In [3]:
chat_model = get_chat_model()
emb_model = get_embedding_model()
topic_selector = get_topic_selector(
    chat_model=ChatOpenAI(
        model="gpt-5-nano",
        temperature=0.1,
        api_key=ENV.llm.api_key
    )
)
db = get_mongo_db()
vector_db = get_mongo_vdb(emb_model, db)
rag_engine = get_rag_engine(vector_db)
topic_prompt_builder = get_topic_prompt_builder(db)

MongoDB connected successfully


In [4]:
agent = get_agent(
    chat_model=chat_model,
    classify_topic=topic_selector,
    rag_retrieve=rag_engine,
    build_topic_prompts=topic_prompt_builder
)  # type: ignore

In [7]:
history = []
query = ""
result = ""

In [8]:
if query and result:
    history.extend([HumanMessage(query), AIMessage(result)])
query = "que puedes hacer?"
result = ""
async for token in agent.stream([*history, HumanMessage(content=query)]):
    if token.type == "info" or token.type == "error":
        display(Markdown(token.content), clear=False)
        continue
    result += token.content
    # display(Markdown(result), clear=True)
display(Markdown(result), clear=False)

Analizando consulta ...

{
  "topic": "CAPABILITIES",
  "description": "Preguntas sobre las funciones y capacidades del asistente.",
  "user_query": "que puedes hacer?",
  "optimized_query": "que puedes hacer",
  "additional_topics": [
    "VERIFICATION_OF_NEWS",
    "QUESTIONS_AND_ANSWERS"
  ]
}


Obteniendo información...

Main documents: 10
Additional topics: [<Topic.VERIFICATION_OF_NEWS: 'VERIFICATION_OF_NEWS'>, <Topic.QUESTIONS_AND_ANSWERS: 'QUESTIONS_AND_ANSWERS'>]


Procesando información...

{ObjectId('68e52b907624707e3849bcc2'), ObjectId('68e52b907624707e3849bc8e'), ObjectId('68e52b907624707e3849bcbd'), ObjectId('68e52b907624707e3849bcc6')}
{ObjectId('68e52b907624707e3849bcc2')}

Eres Checkibot, un asistente especializado en proporcionar información precisa y confiable.

Se ha recuperado la siguiente información del sistema:

# Información encontrada

## Verificaciones encontradas

### 1. Vídeo asegura que “Plan Trabajo” de Áñez es copia de “Plan Empleo” de Goni

Esta noticia fue classificada como [Engañoso](https://chequeabolivia.bo/enganosa)
Fecha de publicación: 04/05/2020 a las 19:48
Resumen: En redes sociales circula un vídeo con la frase “Jeanine Áñez copia plan de Goni”, en el cual se comparan fragmentos del anuncio del “Plan Trabajo”, realizado por la presidente del Estado, con el supuesto “Plan Empleo” de Gonzalo Sánchez de Lozada, para las elecciones presidenciales de 2002. La publicación fue hecha por la página MAS-IPSP Potosí Bolivia. ChequeaBolivia advierte q

Generando respuesta...

Puedo proporcionar información precisa y confiable sobre verificaciones, noticias, contenidos digitales, planes de gobierno y otros temas relacionados. Además, puedo consultar y explicar resultados de investigaciones, verificar la autenticidad de contenidos, ofrecer enlaces y fuentes, y responder a consultas específicas con base en la información recuperada del sistema.

In [ ]:
%rm -rf chroma_db
%cp -r ../chroma_db .
